# 03 — Combined Signal Methods

Three approaches to combining 13 characteristics into a single trading signal:

| Method | Approach | Signal Weights |
|--------|----------|----------------|
| **M1** | Consensus Voting | Each signal casts ±1 vote; trade when net votes exceed threshold |
| **M2** | Equal-Weight Composite | Scale each signal to [-1,1], average, sort on composite |
| **M3** | IC-Weighted Composite | Weight signals by rolling out-of-sample Information Coefficient |

All three methods use the same 13 characteristics with shared percentile cutoffs
and portfolio construction parameters.

In [ ]:
# ── Environment setup (run once) ──────────────────────────────────────
import sys, os

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_PATH = "/content/drive/MyDrive/quant-trading-experiments"
    %pip install -q -e "{REPO_PATH}[dev]"

    JKP_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"
else:
    JKP_CSV_PATH = "/Users/abhin/Library/CloudStorage/GoogleDrive-abhinavkeshri1999@gmail.com/My Drive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"

print(f"Data path: {JKP_CSV_PATH}")
print(f"Exists: {os.path.exists(JKP_CSV_PATH)}")

try:
    import quant_trading
    print(f"quant_trading v{quant_trading.__version__} imported OK")
except ImportError:
    print("ERROR: quant_trading not importable — restart kernel after first run")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from quant_trading.data import load_jkp_csv, winsorize_returns
from quant_trading.strategies import run_method1_consensus_voting, run_composite_method
from quant_trading.plotting import plot_cumulative_returns

In [ ]:
# JKP_CSV_PATH is set in the setup cell above
df_jkp = load_jkp_csv(JKP_CSV_PATH)
df_jkp = winsorize_returns(df_jkp, lower=0.05, upper=0.95)
df_jkp = df_jkp.loc[:, ~df_jkp.columns.duplicated(keep="first")]

# Filter to post-1990 for computational tractability
df_jkp = df_jkp[df_jkp["month_date"] >= pd.Timestamp("1990-01-01")].copy()
print(f"Rows: {len(df_jkp):,}")

## Configuration

In [ ]:
char_specs = [
    {"char_name": "ret_6_1",       "long_high": True},
    {"char_name": "ret_1_0",       "long_high": False},
    {"char_name": "qmj",           "long_high": True},
    {"char_name": "op_at",         "long_high": True},
    {"char_name": "be_me",         "long_high": True},
    {"char_name": "at_gr1",        "long_high": False},
    {"char_name": "beta_60m",      "long_high": False},
    {"char_name": "market_equity", "long_high": False},
    {"char_name": "oaccruals_at",  "long_high": False},
    {"char_name": "dbnetis_at",    "long_high": False},
    {"char_name": "debt_at",       "long_high": False},
    {"char_name": "niq_at_chg1",   "long_high": True},
    {"char_name": "ni_be",         "long_high": True},
]

common_signal_params = {"lower_pct": 0.05, "upper_pct": 0.05, "rebal_period": 1}

consensus_params = {
    "min_net_votes": [2, 4, 7],
    "portfolio_weight_schemes": ["equal"],
}

composite_portfolio_params = {
    "signal_scale_method": ["minmax", "rank"],
    "composite_lower_pct": 0.05,
    "composite_upper_pct": 0.05,
    "portfolio_weight_schemes": ["equal", "signal_rank", "signal_minmax"],
}

weighted_composite_params = {
    "lookback_months": 18,
    "min_history_months": 6,
    "use_abs_ic": False,
    "clip_negative_ic_to_zero": True,
    "auto_direction_from_ic": True,
}

portfolio_construction_params = {"max_weight_per_leg": 0.10}
run_cols = {
    "id_col": "id", "date_col": "month_date",
    "ret_col": "ret_exc_lead1m_w", "weight_col": "market_equity", "freq": 12,
}

print(f"Signals: {len(char_specs)}")

## Run All Methods

In [ ]:
summary_table = pd.DataFrame()
return_series = {}

# --- Method 1: Consensus Voting ---
for n in tqdm(consensus_params["min_net_votes"], desc="Method 1"):
    for ws in consensus_params["portfolio_weight_schemes"]:
        out = run_method1_consensus_voting(
            data=df_jkp, char_specs=char_specs,
            lower_pct=common_signal_params["lower_pct"],
            upper_pct=common_signal_params["upper_pct"],
            rebal_period=common_signal_params["rebal_period"],
            min_net_votes=n, portfolio_weight_scheme=ws,
            ret_col=run_cols["ret_col"], weight_col=run_cols["weight_col"],
            max_weight_per_leg=portfolio_construction_params["max_weight_per_leg"],
            freq=run_cols["freq"],
        )
        key = f"M1 | n={n} | {ws}"
        summary_table = pd.concat([summary_table, out["metrics"]], ignore_index=True)
        return_series[key] = out["returns"]["Spread"]

# --- Method 2: Equal-Weight Composite ---
scale_methods = composite_portfolio_params["signal_scale_method"]
for sm in tqdm(scale_methods, desc="Method 2"):
    for ws in composite_portfolio_params["portfolio_weight_schemes"]:
        out = run_composite_method(
            data=df_jkp, char_specs=char_specs,
            lower_pct=composite_portfolio_params["composite_lower_pct"],
            upper_pct=composite_portfolio_params["composite_upper_pct"],
            rebal_period=common_signal_params["rebal_period"],
            signal_scale_method=sm, signal_weight_method="equal",
            portfolio_weight_scheme=ws,
            ret_col=run_cols["ret_col"], weight_col=run_cols["weight_col"],
            max_weight_per_leg=portfolio_construction_params["max_weight_per_leg"],
            freq=run_cols["freq"],
        )
        key = f"M2 | scale={sm} | {ws}"
        summary_table = pd.concat([summary_table, out["metrics"]], ignore_index=True)
        return_series[key] = out["returns"]["Spread"]

# --- Method 3: IC-Weighted Composite ---
for sm in tqdm(scale_methods, desc="Method 3"):
    for ws in composite_portfolio_params["portfolio_weight_schemes"]:
        out = run_composite_method(
            data=df_jkp, char_specs=char_specs,
            lower_pct=composite_portfolio_params["composite_lower_pct"],
            upper_pct=composite_portfolio_params["composite_upper_pct"],
            rebal_period=common_signal_params["rebal_period"],
            signal_scale_method=sm, signal_weight_method="rolling_ic",
            portfolio_weight_scheme=ws,
            lookback_months=weighted_composite_params["lookback_months"],
            min_history_months=weighted_composite_params["min_history_months"],
            use_abs_ic=weighted_composite_params["use_abs_ic"],
            clip_negative_ic_to_zero=weighted_composite_params["clip_negative_ic_to_zero"],
            auto_direction_from_ic=weighted_composite_params["auto_direction_from_ic"],
            ret_col=run_cols["ret_col"], weight_col=run_cols["weight_col"],
            max_weight_per_leg=portfolio_construction_params["max_weight_per_leg"],
            freq=run_cols["freq"],
        )
        key = f"M3 | scale={sm} | {ws}"
        summary_table = pd.concat([summary_table, out["metrics"]], ignore_index=True)
        return_series[key] = out["returns"]["Spread"]

summary_table = summary_table.sort_values(["Method", "Sharpe"], ascending=[True, False]).reset_index(drop=True)
print(f"\nCompleted: {len(summary_table)} configs")

In [ ]:
display(
    summary_table.style.format({
        "Annualized Mean": "{:.2%}",
        "Annualized Vol": "{:.2%}",
        "Sharpe": "{:.2f}",
        "Max Drawdown": "{:.2%}",
        "Hit Rate": "{:.2%}",
        "Best Month": "{:.2%}",
        "Worst Month": "{:.2%}",
    })
)

In [ ]:
fig = plot_cumulative_returns(return_series, title="Method 1 vs Method 2 vs Method 3")
plt.show()